# 🛡️ BƯỚC 1: HERETIC UNCENSORING PIPELINE (Chạy trên Colab L4 / A100)
Notebook này thực hiện **loại bỏ 100% kiểm duyệt / từ chối trả lời (Censorship / Refusal)** cho các mô hình Coding bằng công cụ **Heretic**.

### 📌 Quy trình:
1. Cài đặt Heretic & dependencies trên môi trường GPU L4/A100.
2. Kết nối Google Drive để lưu model sau khi uncensor.
3. Chạy Heretic tự động tối ưu hóa tham số (Optuna) để triệt tiêu refusal mà giữ nguyên trí thông minh.
4. Xuất và lưu trữ model đã bóc bỏ kiểm duyệt vào Google Drive hoặc tải lên Hugging Face Hub.

In [ ]:
# @title 1. Kiểm tra GPU & Cài đặt Heretic LLM
!nvidia-smi

# Cài đặt Heretic và các thư viện cần thiết
!pip install -U heretic-llm torch torchvision transformers accelerate bitsandbytes huggingface_hub

In [ ]:
# @title 2. Kết nối Google Drive để lưu trữ Model
from google.colab import drive
import os

drive.mount('/content/drive')

# Tạo thư mục chứa các model uncensored trên Drive
SAVE_DIR = "/content/drive/MyDrive/ai_coding_models_uncensored"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"📁 Thư mục lưu trữ model: {SAVE_DIR}")

### 3. Chọn Model để tiến hành Uncensor
Chọn 1 trong 3 model coding cốt lõi:

In [ ]:
# @title 3. Chạy Heretic Uncensoring
# @markdown Chọn model bạn muốn xử lý:
MODEL_CHOICE = "Qwen/Qwen2.5-Coder-7B-Instruct" # @param ["Qwen/Qwen2.5-Coder-7B-Instruct", "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B", "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct", "Qwen/Qwen2.5-Coder-14B-Instruct"]
USE_4BIT_QUANT = True # @param {type:"boolean"}
OUTPUT_MODEL_NAME = "Qwen2.5-Coder-7B-Heretic-Uncensored" # @param {type:"string"}

cmd = f"heretic {MODEL_CHOICE}"
if USE_4BIT_QUANT:
    cmd += " --quantization bnb_4bit"

print(f"🚀 Đang bắt đầu Heretic Uncensor cho: {MODEL_CHOICE}")
print(f"Lệnh thực thi: {cmd}")
!{cmd}

In [ ]:
# @title 4. Lưu Model đã Uncensor vào Google Drive hoặc HuggingFace Hub
# @markdown Sau khi Heretic chạy xong và tạo ra model merged, copy vào Google Drive:
import shutil

LOCAL_MODEL_PATH = "./model-heretic" # Đường dẫn model sau khi merge trong Heretic
TARGET_PATH = os.path.join(SAVE_DIR, OUTPUT_MODEL_NAME)

if os.path.exists(LOCAL_MODEL_PATH):
    print(f"📦 Đang sao chép model sang Google Drive: {TARGET_PATH}...")
    shutil.copytree(LOCAL_MODEL_PATH, TARGET_PATH, dirs_exist_ok=True)
    print("✅ Lưu model vào Google Drive thành công!")
else:
    print("⚠️ Vui lòng hoàn tất bước chạy Heretic trước khi lưu.")

In [ ]:
# @title 5. (Tùy chọn) Đẩy model lên Hugging Face Hub riêng tư / công khai
HF_TOKEN = "" # @param {type:"string"}
HF_REPO_ID = "your-username/Qwen2.5-Coder-7B-Heretic" # @param {type:"string"}

if HF_TOKEN and os.path.exists(TARGET_PATH):
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    print(f"📤 Đang tải model lên Hugging Face: {HF_REPO_ID}...")
    api.create_repo(repo_id=HF_REPO_ID, exist_ok=True, private=True)
    api.upload_folder(
        folder_path=TARGET_PATH,
        repo_id=HF_REPO_ID,
        repo_type="model"
    )
    print("🎉 Upload Hugging Face hoàn tất!")
else:
    print("ℹ️ Bỏ qua bước upload Hugging Face (hoặc chưa nhập token).")